# 17: Same-model controls for the 2015 to 2020 sensitivity drop

Design decisions
- The frozen 2015 stacked ensemble of notebook 07 (two-stage cross-fitted calibration, seed 42, 80/20 stratified split, threshold 0.12) is reused unchanged. A reproduction gate requires it to reproduce the committed notebook-07 predictions on the 2015 test set and on the 2020 transport set before any control is computed.
- The observed drop is decomposed along one fixed model. S0: 2015 test set, all 21 predictors. S1: the same people with the ten predictors absent from the 2020 file replaced by 2015 training medians (the notebook-07 construction). S1w: S1 reweighted so that the eleven shared predictors follow their 2020 distribution. S2: the 2020 file under the same model and imputation. Components: feature availability, shift in the shared-predictor distribution, and change in the outcome given the shared predictors (S1w minus S2). The first two are reported in both orders and as their Shapley average.
- Density-ratio weights come from a five-fold cross-fitted LightGBM classifier separating 2015 test rows from 2020 rows on the eleven shared predictors. 2020 BMI is rounded for this classifier because the 2015 file stores BMI as integers.
- Encoding counterfactuals on the same model and imputation: BMI rounded to integers, alphabetical codes for GenHlth and Diabetes, and the original-submission construction with every categorical column label-encoded.
- Comparison deployments: the same stack with multiple imputation of the ten unavailable predictors (sequential conditional draws from LightGBM classifiers fitted on 2015 training rows, ten draws, predicted probabilities averaged over draws), and a logistic regression fitted on the same 2015 training rows with median imputation. Draws take observed 2015 values only; conditional means were rejected because fractional values of binary predictors are routed differently by the XGBoost, LightGBM and random-forest splits.
- The availability component is attributed to the ten imputed predictors by exact Shapley values over all 1,024 subsets, computed on 2015 test positives.
- Outcome lineage: the processed 2015 and 2020 targets are compared with three constructs derived from the raw CDC BRFSS files: _MICHD (coronary heart disease or myocardial infarction), CVDINFR4 (myocardial infarction only) and CVDCRHD4 (coronary heart disease or angina only), overall and across 26 age-by-sex strata.
- Bootstrap intervals: 1,000 resamples of the 2015 test rows and the 2020 rows, drawn independently and shared across deployments; density-ratio weights and imputation draws are held fixed within resamples.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, math, shutil, zipfile, hashlib, warnings, subprocess, urllib.request
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore', message='X does not have valid feature names')

import numpy as np
import pandas as pd
import joblib
import sklearn, xgboost, lightgbm
import matplotlib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
import tensorflow as tf
from tensorflow import keras

REPO = '/content/drive/MyDrive/EXHEART_Research/exheart-research'
SEED = 42
THRESHOLD = 0.12
B_BOOT = 1000

DATA15 = f'{REPO}/data/brfss2015/heart_disease_health_indicators_BRFSS2015.csv'
DATA20 = f'{REPO}/data/brfss2020/heart_2020_cleaned.csv'
NB07_RESULTS = f'{REPO}/results/editor_revision_fullstack_crossfit'
NB07_MODELS = f'{REPO}/models/editor_revision_fullstack_crossfit/brfss2015'
OUT = f'{REPO}/results/same_model_controls'
FIG = f'{OUT}/figures'
RAW = f'{REPO}/data/raw_cdc'          # CDC archives kept on Drive (never committed)
XPT_DIR = '/content/raw_cdc_xpt'      # extracted XPT files on local runtime disk
for d in (OUT, FIG, RAW, XPT_DIR):
    os.makedirs(d, exist_ok=True)

np.random.seed(SEED)
tf.random.set_seed(SEED)
assert os.path.isfile(f'{REPO}/tools/commit_cell.py'), 'tools/commit_cell.py missing in the repository'

VERSIONS = {'python': sys.version.split()[0], 'numpy': np.__version__, 'pandas': pd.__version__,
            'sklearn': sklearn.__version__, 'xgboost': xgboost.__version__,
            'lightgbm': lightgbm.__version__, 'tensorflow': tf.__version__}
print(json.dumps(VERSIONS, indent=1))

## Data and constructions (identical to notebook 07)

In [ ]:
YESNO = {'No': 0, 'Yes': 1}
GENHEALTH = {'Excellent': 1, 'Very good': 2, 'Good': 3, 'Fair': 4, 'Poor': 5}
AGE_ORDER = ['18-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54',
             '55-59', '60-64', '65-69', '70-74', '75-79', '80 or older']
AGE_MAP = {v: i + 1 for i, v in enumerate(AGE_ORDER)}
DIABETES_SEMANTIC = {'No': 0, 'No, borderline diabetes': 1, 'Yes (during pregnancy)': 1, 'Yes': 2}
SHARED = ['BMI', 'Smoker', 'Stroke', 'PhysHlth', 'MentHlth', 'DiffWalk',
          'Sex', 'Age', 'Diabetes', 'PhysActivity', 'GenHlth']

df15 = pd.read_csv(DATA15)
y15 = df15['HeartDiseaseorAttack'].astype(int).reset_index(drop=True)
X15 = df15.drop(columns=['HeartDiseaseorAttack']).astype(float).reset_index(drop=True)
FEATURES = X15.columns.tolist()
UNAVAILABLE = [c for c in FEATURES if c not in SHARED]
assert len(FEATURES) == 21 and len(UNAVAILABLE) == 10

# Split of notebook 07: row indices, stratified on the outcome, seed 42.
idx_train, idx_test = train_test_split(np.arange(len(y15)), test_size=0.20, stratify=y15, random_state=SEED)
X15_tr = X15.iloc[idx_train].reset_index(drop=True)
X15_te = X15.iloc[idx_test].reset_index(drop=True)
y15_tr = y15.iloc[idx_train].to_numpy()
y15_te = y15.iloc[idx_test].to_numpy()
MEDIANS = X15_tr.median()
assert (X15_te['BMI'] % 1 == 0).all()

df20 = pd.read_csv(DATA20)
y20 = (df20['HeartDisease'] == 'Yes').astype(int).to_numpy()


def alpha_codes(s):
    # LabelEncoder equivalent: categories sorted alphabetically and coded 0..k-1.
    return pd.Categorical(s, categories=sorted(s.unique())).codes.astype(float)


def shared_2020(df, encoding='semantic', round_bmi=False):
    X = pd.DataFrame(index=df.index)
    X['BMI'] = df['BMI'].astype(float).round() if round_bmi else df['BMI'].astype(float)
    X['Smoker'] = df['Smoking'].map(YESNO).astype(float)
    X['Stroke'] = df['Stroke'].map(YESNO).astype(float)
    X['PhysHlth'] = df['PhysicalHealth'].astype(float)
    X['MentHlth'] = df['MentalHealth'].astype(float)
    X['DiffWalk'] = df['DiffWalking'].map(YESNO).astype(float)
    X['Sex'] = df['Sex'].map({'Female': 0, 'Male': 1}).astype(float)
    X['PhysActivity'] = df['PhysicalActivity'].map(YESNO).astype(float)
    if encoding == 'semantic':
        X['Age'] = df['AgeCategory'].map(AGE_MAP).astype(float)
        X['Diabetes'] = df['Diabetic'].map(DIABETES_SEMANTIC).astype(float)
        X['GenHlth'] = df['GenHealth'].map(GENHEALTH).astype(float)
    elif encoding == 'blind_genhlth_diabetes':
        X['Age'] = df['AgeCategory'].map(AGE_MAP).astype(float)
        X['Diabetes'] = alpha_codes(df['Diabetic'])
        X['GenHlth'] = alpha_codes(df['GenHealth'])
    elif encoding == 'labelencoder_all':
        # Construction of the original submission: every categorical column label-encoded alphabetically.
        X['Age'] = alpha_codes(df['AgeCategory'])
        X['Diabetes'] = alpha_codes(df['Diabetic'])
        X['GenHlth'] = alpha_codes(df['GenHealth'])
        X['Sex'] = alpha_codes(df['Sex'])
        for src, dst in [('Smoking', 'Smoker'), ('Stroke', 'Stroke'),
                         ('DiffWalking', 'DiffWalk'), ('PhysicalActivity', 'PhysActivity')]:
            X[dst] = alpha_codes(df[src])
    else:
        raise ValueError(encoding)
    assert X.notna().all().all(), encoding
    return X[SHARED]


ENCODINGS = ['bmi_rounded', 'blind_genhlth_diabetes', 'labelencoder_all']
S20 = {'semantic': shared_2020(df20),
       'bmi_rounded': shared_2020(df20, round_bmi=True),
       'blind_genhlth_diabetes': shared_2020(df20, 'blind_genhlth_diabetes'),
       'labelencoder_all': shared_2020(df20, 'labelencoder_all')}


def impute_median(shared_df):
    X = shared_df[SHARED].copy()
    for c in UNAVAILABLE:
        X[c] = float(MEDIANS[c])
    return X[FEATURES]


print('2015 test rows', len(y15_te), 'positives', int(y15_te.sum()), '| 2020 rows', len(y20), 'positives', int(y20.sum()))
print('unavailable in 2020:', UNAVAILABLE)
print('2015 training medians:', {c: float(MEDIANS[c]) for c in UNAVAILABLE})

## Metric helpers

In [ ]:
def _weights(y, w):
    return np.ones(len(y)) if w is None else np.asarray(w, float)


def sensitivity(y, p, w=None, t=THRESHOLD):
    y = np.asarray(y); p = np.asarray(p); w = _weights(y, w); pos = y == 1
    return float((w[pos] * (p[pos] >= t)).sum() / w[pos].sum())


def specificity(y, p, w=None, t=THRESHOLD):
    y = np.asarray(y); p = np.asarray(p); w = _weights(y, w); neg = y == 0
    return float((w[neg] * (p[neg] < t)).sum() / w[neg].sum())


def ece(y, p, w=None, n_bins=10):
    # Binning of notebook 07 (ten equal-width bins, last bin closed), optionally weighted.
    y = np.asarray(y, float); p = np.asarray(p, float); w = _weights(y, w)
    edges = np.linspace(0, 1, n_bins + 1)
    total = 0.0
    for i in range(n_bins):
        upper = (p <= edges[i + 1]) if i == n_bins - 1 else (p < edges[i + 1])
        m = (p >= edges[i]) & upper
        if m.any():
            total += w[m].sum() * abs(np.average(y[m], weights=w[m]) - np.average(p[m], weights=w[m]))
    return float(total / w.sum())


def summarise(y, p, w=None):
    y = np.asarray(y); p = np.asarray(p); ww = _weights(y, w)
    return {'n': int(len(y)),
            'prevalence': float(np.average(y, weights=ww)),
            'AUC': float(roc_auc_score(y, p, sample_weight=None if w is None else ww)),
            'Sensitivity': sensitivity(y, p, w),
            'Specificity': specificity(y, p, w),
            'ECE': ece(y, p, w),
            'mean_probability': float(np.average(p, weights=ww)),
            'selection_rate': float(np.average(p >= THRESHOLD, weights=ww))}

## Frozen notebook-07 stack and reproduction gate

In [ ]:
REQUIRED = ['xgb.pkl', 'lgbm.pkl', 'rf.pkl', 'scaler.pkl', 'meta.pkl', 'platt.pkl', 'mlp.keras']
missing = [f for f in REQUIRED if not os.path.isfile(f'{NB07_MODELS}/{f}')]
assert not missing, f'notebook-07 model files missing from {NB07_MODELS}: {missing}'

STACK = {k: joblib.load(f'{NB07_MODELS}/{k}.pkl') for k in ['xgb', 'lgbm', 'rf', 'scaler', 'meta', 'platt']}
STACK['mlp'] = keras.models.load_model(f'{NB07_MODELS}/mlp.keras', compile=False)


def predict_stack(X):
    # Prediction path of notebook 07 (predict_with_fitted_stack): float32 inputs, scaler for the MLP only.
    A = pd.DataFrame(X, columns=FEATURES).to_numpy(dtype=np.float32)
    base = np.column_stack([
        STACK['xgb'].predict_proba(A)[:, 1],
        STACK['lgbm'].predict_proba(A)[:, 1],
        STACK['rf'].predict_proba(A)[:, 1],
        STACK['mlp'].predict(STACK['scaler'].transform(A), batch_size=4096, verbose=0).ravel(),
    ])
    raw = STACK['meta'].predict_proba(base)[:, 1]
    return STACK['platt'].predict_proba(raw.reshape(-1, 1))[:, 1]


ref15 = pd.read_csv(f'{NB07_RESULTS}/brfss2015_test_predictions.csv')
ref20 = pd.read_csv(f'{NB07_RESULTS}/brfss2020_transport_predictions.csv')
p_full15 = predict_stack(X15_te)
p_transport20 = predict_stack(impute_median(S20['semantic']))

GATE = {
    'labels_2015_identical': bool(np.array_equal(ref15['y_true'].to_numpy(), y15_te)),
    'sex_2015_identical': bool(np.array_equal(ref15['Sex'].to_numpy(), X15_te['Sex'].to_numpy())),
    'labels_2020_identical': bool(np.array_equal(ref20['y_true'].to_numpy(), y20)),
    'max_abs_prob_diff_2015': float(np.max(np.abs(p_full15 - ref15['prob_calibrated'].to_numpy()))),
    'max_abs_prob_diff_2020': float(np.max(np.abs(p_transport20 - ref20['prob_calibrated'].to_numpy()))),
    'sensitivity_2015_committed': sensitivity(y15_te, ref15['prob_calibrated']),
    'sensitivity_2015_reproduced': sensitivity(y15_te, p_full15),
    'sensitivity_2020_committed': sensitivity(y20, ref20['prob_calibrated']),
    'sensitivity_2020_reproduced': sensitivity(y20, p_transport20),
    'tolerance_probability': 1e-4,
    'tolerance_sensitivity': 5e-4,
    'versions': VERSIONS,
}
GATE['passed'] = bool(
    GATE['labels_2015_identical'] and GATE['sex_2015_identical'] and GATE['labels_2020_identical']
    and GATE['max_abs_prob_diff_2015'] <= GATE['tolerance_probability']
    and GATE['max_abs_prob_diff_2020'] <= GATE['tolerance_probability']
    and abs(GATE['sensitivity_2015_committed'] - GATE['sensitivity_2015_reproduced']) <= GATE['tolerance_sensitivity']
    and abs(GATE['sensitivity_2020_committed'] - GATE['sensitivity_2020_reproduced']) <= GATE['tolerance_sensitivity'])
json.dump(GATE, open(f'{OUT}/reproduction_check.json', 'w'), indent=2)
print(json.dumps(GATE, indent=1))
if not GATE['passed']:
    raise RuntimeError('Frozen notebook-07 models do not reproduce the committed predictions; controls not computed.')

## Comparison deployments: logistic regression and multiple imputation

In [ ]:
# Logistic regression on the same 2015 training rows; unweighted fit, so its probabilities are on the
# outcome scale and the 0.12 threshold keeps its meaning without a separate calibrator.
LR_SCALER = StandardScaler().fit(X15_tr.to_numpy(float))
LOGREG = LogisticRegression(max_iter=5000).fit(LR_SCALER.transform(X15_tr.to_numpy(float)), y15_tr)


def predict_logistic(X):
    A = pd.DataFrame(X, columns=FEATURES).to_numpy(float)
    return LOGREG.predict_proba(LR_SCALER.transform(A))[:, 1]


# Multiple imputation of the ten unavailable predictors by sequential conditional draws:
# predictor j is drawn from P(U_j | shared predictors, U_1..U_{j-1}), estimated by a LightGBM classifier
# on 2015 training rows. Every draw is an observed 2015 value, so tree splits see in-support inputs.
M_IMP = 10
IMP_MODELS = []
for j, c in enumerate(UNAVAILABLE):
    cols = SHARED + UNAVAILABLE[:j]
    clf = LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31, min_child_samples=50,
                         random_state=SEED, verbose=-1)
    clf.fit(X15_tr[cols].to_numpy(float), X15_tr[c].to_numpy().astype(int))
    IMP_MODELS.append((c, cols, clf))


def impute_multiple(shared_df, seed):
    rng = np.random.default_rng(seed)
    X = shared_df[SHARED].copy()
    for c, cols, clf in IMP_MODELS:
        P = clf.predict_proba(X[cols].to_numpy(float))
        u = rng.random(len(X))
        k = (P.cumsum(axis=1) < u[:, None]).sum(axis=1)
        X[c] = clf.classes_[np.minimum(k, P.shape[1] - 1)].astype(float)
    return X[FEATURES]


def predict_stack_multiple(shared_df):
    return np.mean([predict_stack(impute_multiple(shared_df, SEED + m)) for m in range(M_IMP)], axis=0)


# Imputer quality on 2015 test rows (observed earlier predictors as inputs) and marginal check of one draw.
one_draw = impute_multiple(X15_te[SHARED], SEED)
quality = []
for c, cols, clf in IMP_MODELS:
    P = clf.predict_proba(X15_te[cols].to_numpy(float))
    obs = X15_te[c].to_numpy(float)
    row = {'feature': c, 'n_classes': int(len(clf.classes_)),
           'mean_observed_2015_test': float(obs.mean()), 'mean_one_draw_2015_test': float(one_draw[c].mean())}
    if len(clf.classes_) == 2:
        row['auc_2015_test'] = float(roc_auc_score(obs, P[:, 1]))
    else:
        row['spearman_expected_vs_observed_2015_test'] = float(
            pd.Series(P @ clf.classes_.astype(float)).corr(pd.Series(obs), method='spearman'))
    quality.append(row)
IMPUTER_QUALITY = pd.DataFrame(quality)
IMPUTER_QUALITY.to_csv(f'{OUT}/multiple_imputation_quality.csv', index=False)
print(IMPUTER_QUALITY.round(3).to_string(index=False))
print('logistic regression 2015 test:', {k: round(v, 4) for k, v in summarise(y15_te, predict_logistic(X15_te)).items()})

## Predictions under every condition

In [ ]:
DEPLOYMENTS = {
    'stack_median': lambda S: predict_stack(impute_median(S)),
    'stack_multiple': predict_stack_multiple,
    'logistic_median': lambda S: predict_logistic(impute_median(S)),
}
FULL15 = {'stack': p_full15, 'logistic': predict_logistic(X15_te)}

PRED = {}
for name, deploy in DEPLOYMENTS.items():
    t0 = time.time()
    PRED[(name, 'S0')] = FULL15[name.split('_')[0]]
    PRED[(name, 'S1')] = deploy(X15_te[SHARED])
    PRED[(name, 'S2')] = p_transport20 if name == 'stack_median' else deploy(S20['semantic'])
    print(f'{name}: {time.time() - t0:.0f} s')
for enc in ENCODINGS:
    PRED[('stack_median', f'S2_{enc}')] = predict_stack(impute_median(S20[enc]))

for (name, cond), p in PRED.items():
    y = y15_te if cond in ('S0', 'S1') else y20
    print(f'{name:18s} {cond:28s} sensitivity {sensitivity(y, p):.4f}  AUC {roc_auc_score(y, p):.4f}')

## Density-ratio weights for the shared-predictor distribution

In [ ]:
D15 = X15_te[SHARED].copy()
D20 = S20['bmi_rounded'].copy()   # 2015 BMI is integer-valued; rounding 2020 BMI stops precision identifying the wave
Z = pd.concat([D15, D20], ignore_index=True).to_numpy(float)
dom = np.r_[np.zeros(len(D15)), np.ones(len(D20))].astype(int)

oof = np.zeros(len(dom))
for k, (tr, va) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(Z, dom)):
    clf = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=100,
                         subsample=0.8, subsample_freq=1, colsample_bytree=0.9,
                         random_state=SEED + k, verbose=-1)
    clf.fit(Z[tr], dom[tr])
    oof[va] = clf.predict_proba(Z[va])[:, 1]

p_dom = np.clip(oof[:len(D15)], 1e-6, 1 - 1e-6)
W_RAW = p_dom / (1 - p_dom) * (len(D15) / len(D20))
W_CAP = float(np.quantile(W_RAW, 0.999))
W = np.minimum(W_RAW, W_CAP)


def ess(w):
    return float(w.sum() ** 2 / (w ** 2).sum())


balance = []
for c in SHARED:
    a, b = D15[c].to_numpy(float), D20[c].to_numpy(float)
    sd = math.sqrt((a.var() + b.var()) / 2) or 1.0
    balance.append({'feature': c, 'mean_2015_test': a.mean(), 'mean_2015_test_reweighted': np.average(a, weights=W),
                    'mean_2020': b.mean(), 'smd_before': (a.mean() - b.mean()) / sd,
                    'smd_after': (np.average(a, weights=W) - b.mean()) / sd})
BALANCE = pd.DataFrame(balance)
BALANCE.to_csv(f'{OUT}/covariate_balance.csv', index=False)

WEIGHT_SUMMARY = {
    'domain_auc_cross_fitted': float(roc_auc_score(dom, oof)),
    'weight_cap_quantile_0_999': W_CAP,
    'n_weights_capped': int((W_RAW > W_CAP).sum()),
    'ess_all_2015_test': ess(W), 'n_2015_test': int(len(W)),
    'ess_positives_2015_test': ess(W[y15_te == 1]), 'n_positives_2015_test': int((y15_te == 1).sum()),
    'max_abs_smd_before': float(BALANCE['smd_before'].abs().max()),
    'max_abs_smd_after': float(BALANCE['smd_after'].abs().max()),
    'prevalence_2015_test': float(y15_te.mean()),
    'prevalence_2015_test_reweighted': float(np.average(y15_te, weights=W)),
    'prevalence_2020': float(y20.mean()),
}
WEIGHT_SUMMARY['balance_ok'] = bool(WEIGHT_SUMMARY['max_abs_smd_after'] < 0.05)
json.dump(WEIGHT_SUMMARY, open(f'{OUT}/density_ratio_summary.json', 'w'), indent=2)
print(BALANCE.round(4).to_string(index=False))
print(json.dumps(WEIGHT_SUMMARY, indent=1))

## Decomposition of the drop with bootstrap intervals

In [ ]:
POS15 = (y15_te == 1).astype(float)
POS20 = (y20 == 1).astype(float)
HIT = {key: (p >= THRESHOLD).astype(float) * (POS15 if key[1] in ('S0', 'S1') else POS20)
       for key, p in PRED.items()}


def sens_counts(cnt, hit, pos, w=None):
    ww = cnt if w is None else cnt * w
    return float((ww * hit).sum() / (ww * pos).sum())


def components(name, c15, c20):
    S0 = sens_counts(c15, HIT[(name, 'S0')], POS15)
    S0w = sens_counts(c15, HIT[(name, 'S0')], POS15, W)
    S1 = sens_counts(c15, HIT[(name, 'S1')], POS15)
    S1w = sens_counts(c15, HIT[(name, 'S1')], POS15, W)
    S2 = sens_counts(c20, HIT[(name, 'S2')], POS20)
    out = {'S0': S0, 'S0w': S0w, 'S1': S1, 'S1w': S1w, 'S2': S2, 'total_drop': S0 - S2,
           'availability_first': S0 - S1, 'covariate_second': S1 - S1w,
           'covariate_first': S0 - S0w, 'availability_second': S0w - S1w,
           'availability_shapley': ((S0 - S1) + (S0w - S1w)) / 2,
           'covariate_shapley': ((S1 - S1w) + (S0 - S0w)) / 2,
           'conditional': S1w - S2}
    if name == 'stack_median':
        for enc in ENCODINGS:
            out[f'encoding_{enc}'] = S2 - sens_counts(c20, HIT[(name, f'S2_{enc}')], POS20)
    return out


ones15, ones20 = np.ones(len(y15_te)), np.ones(len(y20))
POINT = {name: components(name, ones15, ones20) for name in DEPLOYMENTS}

rng = np.random.default_rng(SEED)
BOOT = {name: [] for name in DEPLOYMENTS}
for b in range(B_BOOT):
    c15 = np.bincount(rng.integers(0, len(y15_te), len(y15_te)), minlength=len(y15_te)).astype(float)
    c20 = np.bincount(rng.integers(0, len(y20), len(y20)), minlength=len(y20)).astype(float)
    for name in DEPLOYMENTS:
        BOOT[name].append(components(name, c15, c20))

rows = []
for name in DEPLOYMENTS:
    bt = pd.DataFrame(BOOT[name])
    total = POINT[name]['total_drop']
    for q, v in POINT[name].items():
        lo, hi = np.quantile(bt[q], [0.025, 0.975])
        share = v / total if (q.startswith(('availability', 'covariate', 'conditional', 'encoding'))
                              and abs(total) > 1e-9) else np.nan
        rows.append({'deployment': name, 'quantity': q, 'estimate': v, 'ci_low': lo, 'ci_high': hi,
                     'share_of_total_drop': share})
DECOMP = pd.DataFrame(rows)
DECOMP.to_csv(f'{OUT}/decomposition.csv', index=False)

cond_rows = []
for (name, cond), p in PRED.items():
    y = y15_te if cond in ('S0', 'S1') else y20
    cond_rows.append({'deployment': name, 'condition': cond, 'weighted_to_2020_shared': False, **summarise(y, p)})
    if cond in ('S0', 'S1'):
        cond_rows.append({'deployment': name, 'condition': cond + 'w', 'weighted_to_2020_shared': True,
                          **summarise(y15_te, p, W)})
CONDITIONS = pd.DataFrame(cond_rows)
CONDITIONS.to_csv(f'{OUT}/conditions.csv', index=False)

show = ['S0', 'S1', 'S1w', 'S2', 'total_drop', 'availability_shapley', 'covariate_shapley', 'conditional',
        'availability_first', 'covariate_second', 'covariate_first', 'availability_second'] + \
       [f'encoding_{e}' for e in ENCODINGS]
print(DECOMP[DECOMP['quantity'].isin(show)].round(4).to_string(index=False))
print()
print(CONDITIONS.round(4).to_string(index=False))

## Attribution of the availability component to the ten imputed predictors (exact Shapley values)

In [ ]:
N_U = len(UNAVAILABLE)
SHAPLEY_WEIGHT = [math.factorial(k) * math.factorial(N_U - k - 1) / math.factorial(N_U) for k in range(N_U)]
POS_X = X15_te[y15_te == 1].reset_index(drop=True)


def subset_sensitivities(predict, chunk=32):
    # v[mask]: sensitivity on 2015 test positives when the predictors in mask are replaced by 2015 training medians.
    v = np.zeros(2 ** N_U)
    for start in range(0, 2 ** N_U, chunk):
        masks = list(range(start, min(start + chunk, 2 ** N_U)))
        frames = []
        for mask in masks:
            Xm = POS_X.copy()
            for j in range(N_U):
                if mask >> j & 1:
                    Xm[UNAVAILABLE[j]] = float(MEDIANS[UNAVAILABLE[j]])
            frames.append(Xm)
        pr = predict(pd.concat(frames, ignore_index=True)).reshape(len(masks), len(POS_X))
        v[start:start + len(masks)] = (pr >= THRESHOLD).mean(axis=1)
    return v


def shapley(v):
    phi = np.zeros(N_U)
    for mask in range(2 ** N_U):
        k = bin(mask).count('1')
        for j in range(N_U):
            if not mask >> j & 1:
                phi[j] += SHAPLEY_WEIGHT[k] * (v[mask] - v[mask | (1 << j)])
    return phi


V = {'stack': subset_sensitivities(predict_stack), 'logistic': subset_sensitivities(predict_logistic)}
tol = 2.0 / len(POS_X)
for fam, dep in [('stack', 'stack_median'), ('logistic', 'logistic_median')]:
    assert abs(V[fam][0] - POINT[dep]['S0']) <= tol and abs(V[fam][-1] - POINT[dep]['S1']) <= tol, fam

PHI = {fam: shapley(V[fam]) for fam in V}
for fam in PHI:
    assert abs(PHI[fam].sum() - (V[fam][0] - V[fam][-1])) < 1e-9

FEATURE_NAMES = {'HighBP': 'High blood pressure', 'HighChol': 'High cholesterol',
                 'CholCheck': 'Cholesterol check in 5 years', 'Fruits': 'Fruit daily',
                 'Veggies': 'Vegetables daily', 'HvyAlcoholConsump': 'Heavy alcohol use',
                 'AnyHealthcare': 'Any health coverage', 'NoDocbcCost': 'Doctor visit missed for cost',
                 'Education': 'Education level', 'Income': 'Income level'}
SHAP_TABLE = pd.DataFrame({
    'feature': UNAVAILABLE, 'paper_name': [FEATURE_NAMES[c] for c in UNAVAILABLE],
    'shapley_stack': PHI['stack'], 'share_stack': PHI['stack'] / PHI['stack'].sum(),
    'shapley_logistic': PHI['logistic'], 'share_logistic': PHI['logistic'] / PHI['logistic'].sum(),
}).sort_values('shapley_stack', ascending=False)
SHAP_TABLE.to_csv(f'{OUT}/availability_shapley.csv', index=False)
pd.DataFrame({'mask': np.arange(2 ** N_U),
              'imputed': ['+'.join(UNAVAILABLE[j] for j in range(N_U) if m >> j & 1) for m in range(2 ** N_U)],
              'sensitivity_stack': V['stack'], 'sensitivity_logistic': V['logistic']}
             ).to_csv(f'{OUT}/availability_subset_sensitivities.csv', index=False)
print(SHAP_TABLE.round(4).to_string(index=False))

## Figures

In [ ]:
matplotlib.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.titlesize': 8.5,
                            'axes.labelsize': 8, 'xtick.labelsize': 7, 'ytick.labelsize': 7,
                            'axes.spines.top': False, 'axes.spines.right': False,
                            'axes.edgecolor': '#52514e', 'axes.labelcolor': '#0b0b0b',
                            'xtick.color': '#52514e', 'ytick.color': '#52514e'})
INK, INK2, GRID, ENDPOINT = '#0b0b0b', '#52514e', '#e6e5e1', '#8c8a84'
C_AVAIL, C_COV, C_COND = '#2a78d6', '#eb6834', '#1baf7a'
DEPLOY_NAMES = {'stack_median': 'Stacked ensemble, median imputation',
                'stack_multiple': 'Stacked ensemble, multiple imputation (10 draws)',
                'logistic_median': 'Logistic regression, median imputation'}
from matplotlib.transforms import blended_transform_factory
from matplotlib.ticker import MaxNLocator, FormatStrFormatter


def save(fig, stem):
    fig.savefig(f'{FIG}/{stem}.png', dpi=600, bbox_inches='tight')
    fig.savefig(f'{FIG}/{stem}.tiff', dpi=600, bbox_inches='tight', pil_kwargs={'compression': 'tiff_lzw'})
    plt.show()


def ci(name, q):
    r = DECOMP[(DECOMP.deployment == name) & (DECOMP.quantity == q)].iloc[0]
    return r['estimate'], r['ci_low'], r['ci_high']


def num(v, signed=False):
    v = 0.0 if abs(v) < 5e-4 else v
    return (f'{v:+.3f}' if signed else f'{v:.3f}').replace('-', '\u2212')


# Figure: same-model decomposition; components are the Shapley averages of the two orders.
ROWS = [('total_drop', 'Total change', ENDPOINT),
        ('availability_shapley', 'Missing predictors', C_AVAIL),
        ('covariate_shapley', 'Shared-predictor shift', C_COV),
        ('conditional', 'Outcome given shared predictors', C_COND)]
changes = [-x for name in DEPLOYMENTS for q, _, _ in ROWS for x in ci(name, q)]
x_lo, x_hi = min(changes + [0.0]), max(changes + [0.0])
pad = 0.06 * (x_hi - x_lo)
fig, axes = plt.subplots(len(DEPLOYMENTS), 1, figsize=(7.2, 1.65 * len(DEPLOYMENTS) + 0.6), sharex=True)
for ax, name in zip(axes, DEPLOYMENTS):
    tr = blended_transform_factory(ax.transAxes, ax.transData)
    for r, (q, label, col) in enumerate(ROWS):
        est, lo, hi = ci(name, q)
        ax.barh(r, -est, height=0.62, color=col, zorder=2)
        ax.errorbar(-est, r, xerr=[[hi - est], [est - lo]], fmt='none', ecolor=INK, elinewidth=0.7,
                    capsize=1.8, capthick=0.7, zorder=3)
        ax.text(1.015, r, f'{num(-est, True)}  [{num(-hi, True)}, {num(-lo, True)}]', transform=tr,
                ha='left', va='center', color=INK, fontsize=6.5)
    ax.axvline(0, color=INK2, lw=0.6, zorder=1)
    ax.set_yticks(range(len(ROWS)))
    ax.set_yticklabels([r[1] for r in ROWS])
    ax.set_ylim(len(ROWS) - 0.5, -0.5)
    ax.tick_params(axis='y', length=0)
    ax.xaxis.grid(True, color=GRID, lw=0.6, zorder=0)
    ax.set_title(f"{DEPLOY_NAMES[name]}\n2015 test {num(POINT[name]['S0'])}, 2020 transport {num(POINT[name]['S2'])}",
                 loc='left', color=INK, linespacing=1.3)
    ax.text(1.015, -0.62, 'Estimate  [95% CI]', transform=tr, ha='left', va='bottom', color=INK2, fontsize=6.5)
axes[-1].set_xlim(x_lo - pad, x_hi + pad)
axes[-1].set_xlabel('Change in sensitivity at threshold 0.12, 2015 test to 2020 transport')
fig.subplots_adjust(left=0.25, right=0.76, top=0.92, bottom=0.09, hspace=0.85)
save(fig, 'fig_same_model_decomposition')

# Figure: Shapley attribution of the availability component.
order = SHAP_TABLE['feature'].tolist()
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0), sharey=True)
for ax, fam, title in [(axes[0], 'stack', 'Stacked ensemble'), (axes[1], 'logistic', 'Logistic regression')]:
    vals = SHAP_TABLE.set_index('feature').loc[order, f'shapley_{fam}'].to_numpy()
    span = max(abs(vals).max(), 0.01)
    ax.barh(range(len(order)), vals, height=0.62, color=C_AVAIL, zorder=2)
    for k, v in enumerate(vals):
        right = v >= -5e-4
        ax.text(v + (0.02 if right else -0.02) * span, k, num(v), va='center',
                ha='left' if right else 'right', color=INK, fontsize=6.5)
    ax.axvline(0, color=INK2, lw=0.6, zorder=1)
    ax.set_title(f'{title} (total {num(PHI[fam].sum())})', color=INK)
    ax.xaxis.grid(True, color=GRID, lw=0.6, zorder=0)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    ax.set_xlabel('Sensitivity lost when the predictor is unavailable\n(Shapley value)')
    ax.set_xlim(min(0, vals.min()) - 0.30 * span, vals.max() + 0.30 * span)
axes[0].set_yticks(range(len(order)))
axes[0].set_yticklabels([FEATURE_NAMES[c] for c in order])
axes[0].set_ylim(len(order) - 0.5, -0.5)
axes[0].tick_params(axis='y', length=0)
fig.tight_layout()
save(fig, 'fig_availability_shapley')

# Figure: encoding counterfactuals on the same model and imputation.
labels = ['Semantic encoding (reported transport)', 'BMI rounded to integers',
          'Alphabetical codes for general health and diabetes', 'All categorical columns label-encoded\n(original submission)']
keys = ['S2'] + [f'S2_{e}' for e in ENCODINGS]
vals = [sensitivity(y20, PRED[('stack_median', k)]) for k in keys]
S0_ref = POINT['stack_median']['S0']
fig, ax = plt.subplots(figsize=(6.2, 2.4))
ax.barh(range(4), vals, height=0.6, color=ENDPOINT, zorder=2)
for k, v in enumerate(vals):
    ax.text(v + 0.012, k, num(v), ha='left', va='center', color=INK, fontsize=7)
ax.axvline(S0_ref, color=INK2, lw=0.8, ls='--', zorder=1)
ax.text(S0_ref, -0.85, f'2015 test, all predictors ({num(S0_ref)})', ha='center', va='center',
        color=INK2, fontsize=6.5, bbox={'facecolor': 'white', 'edgecolor': 'none', 'pad': 1.0})
ax.set_yticks(range(4)); ax.set_yticklabels(labels)
ax.set_ylim(3.5, -1.1)
ax.tick_params(axis='y', length=0)
ax.set_xlim(0, 1.0); ax.set_xlabel('2020 sensitivity at threshold 0.12')
ax.xaxis.grid(True, color=GRID, lw=0.6, zorder=0)
fig.tight_layout()
save(fig, 'fig_encoding_counterfactuals')

## Outcome lineage against the raw CDC BRFSS files

In [ ]:
RAW_URLS = {2015: 'https://www.cdc.gov/brfss/annual_data/2015/files/LLCP2015XPT.zip',
            2020: 'https://www.cdc.gov/brfss/annual_data/2020/files/LLCP2020XPT.zip'}
CONSTRUCTS = {'_MICHD': 'CHD or MI (_MICHD)', 'CVDINFR4': 'MI only (CVDINFR4)', 'CVDCRHD4': 'CHD or angina only (CVDCRHD4)'}


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


def fetch_xpt(year):
    zpath = f'{RAW}/LLCP{year}XPT.zip'
    if not (os.path.isfile(zpath) and zipfile.is_zipfile(zpath)):
        req = urllib.request.Request(RAW_URLS[year], headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=900) as r, open(zpath + '.part', 'wb') as f:
            shutil.copyfileobj(r, f, length=1 << 20)
        os.replace(zpath + '.part', zpath)
    xpath = f'{XPT_DIR}/LLCP{year}.XPT'
    if not os.path.isfile(xpath):
        with zipfile.ZipFile(zpath) as z:
            # The 2015 archive stores its member as 'LLCP2015.XPT ' with a trailing space.
            member = [m for m in z.namelist() if m.strip().upper().endswith('.XPT')][0]
            with z.open(member) as src, open(xpath + '.part', 'wb') as dst:
                shutil.copyfileobj(src, dst, length=1 << 20)
        os.replace(xpath + '.part', xpath)
    return xpath, zpath


def read_raw(year):
    import pyreadstat
    xpath, zpath = fetch_xpt(year)
    _, meta = pyreadstat.read_xport(xpath, metadataonly=True)
    names = list(meta.column_names)
    sexcol = next(c for c in ['_SEX', 'SEXVAR', 'SEX'] if c in names)
    cols = list(CONSTRUCTS) + ['_AGEG5YR', sexcol]
    absent = [c for c in cols if c not in names]
    assert not absent, f'{year}: raw variables absent {absent}'
    raw, _ = pyreadstat.read_xport(xpath, usecols=cols)
    out = pd.DataFrame({CONSTRUCTS[c]: raw[c].map({1: 1, 2: 0}) for c in CONSTRUCTS})
    out['Age'] = raw['_AGEG5YR'].where(raw['_AGEG5YR'].between(1, 13))
    out['Sex'] = raw[sexcol].map({1: 1, 2: 0})
    return out, {'zip_sha256': sha256(zpath), 'raw_rows': int(len(raw)), 'sex_variable': sexcol,
                 'source_url': RAW_URLS[year]}


def stratum_profile(df, col):
    g = df.dropna(subset=[col, 'Age', 'Sex']).groupby(['Age', 'Sex'])[col]
    return g.mean(), g.size()


PROCESSED = {2015: pd.DataFrame({'target': y15, 'Age': X15['Age'], 'Sex': X15['Sex']}),
             2020: pd.DataFrame({'target': y20, 'Age': df20['AgeCategory'].map(AGE_MAP),
                                 'Sex': df20['Sex'].map({'Female': 0, 'Male': 1})})}
LINEAGE = {'status': 'not run'}
lineage_rows = []
try:
    try:
        import pyreadstat
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyreadstat'], check=True)
    for year in (2015, 2020):
        raw, info = read_raw(year)
        proc_prev, proc_n = stratum_profile(PROCESSED[year], 'target')
        names = list(CONSTRUCTS.values())
        complete = raw.dropna(subset=names)
        info['michd_equals_mi_or_chd'] = float(
            ((complete[names[1]] == 1) | (complete[names[2]] == 1)).astype(int).eq(complete[names[0]]).mean())
        for c in names:
            raw_prev, raw_n = stratum_profile(raw, c)
            common = proc_prev.index.intersection(raw_prev.index)
            lineage_rows.append({
                'year': year, 'construct': c, 'n_raw_valid': int(raw[c].notna().sum()),
                'prevalence_raw_unweighted': float(raw[c].mean()),
                'prevalence_processed': float(PROCESSED[year]['target'].mean()),
                'n_processed': int(len(PROCESSED[year])), 'strata_compared': int(len(common)),
                'age_sex_profile_mean_abs_diff': float(np.average(np.abs(proc_prev[common] - raw_prev[common]),
                                                                   weights=proc_n[common]))})
        LINEAGE[year] = info
    LIN = pd.DataFrame(lineage_rows)
    for year in (2015, 2020):
        part = LIN[LIN.year == year].sort_values('age_sex_profile_mean_abs_diff')
        best, second = part.iloc[0], part.iloc[1]
        ratio = second['age_sex_profile_mean_abs_diff'] / max(best['age_sex_profile_mean_abs_diff'], 1e-12)
        LINEAGE[year].update({'closest_construct': best['construct'],
                              'closest_mean_abs_diff': float(best['age_sex_profile_mean_abs_diff']),
                              'next_closest_ratio': float(ratio),
                              'matched': bool(best['age_sex_profile_mean_abs_diff'] <= 0.01 and ratio >= 3)})
    LINEAGE['same_construct_both_years'] = bool(
        LINEAGE[2015]['matched'] and LINEAGE[2020]['matched']
        and LINEAGE[2015]['closest_construct'] == LINEAGE[2020]['closest_construct'])
    LINEAGE['status'] = 'completed'
    LIN.to_csv(f'{OUT}/outcome_lineage.csv', index=False)
    print(LIN.round(4).to_string(index=False))
except Exception as e:
    LINEAGE = {'status': 'failed', 'error': repr(e)}
json.dump({str(k): v for k, v in LINEAGE.items()}, open(f'{OUT}/outcome_lineage.json', 'w'), indent=2)
print(json.dumps({str(k): v for k, v in LINEAGE.items()}, indent=1))

## Summary

In [ ]:
SUMMARY = {'reproduction_gate_passed': GATE['passed'],
           'density_ratio': WEIGHT_SUMMARY,
           'outcome_lineage_status': LINEAGE.get('status'),
           'outcome_same_construct_both_years': LINEAGE.get('same_construct_both_years'),
           'deployments': {}}
for name in DEPLOYMENTS:
    d = DECOMP[DECOMP.deployment == name].set_index('quantity')
    SUMMARY['deployments'][name] = {
        q: {'estimate': float(d.loc[q, 'estimate']), 'ci95': [float(d.loc[q, 'ci_low']), float(d.loc[q, 'ci_high'])],
            'share_of_total_drop': None if pd.isna(d.loc[q, 'share_of_total_drop']) else float(d.loc[q, 'share_of_total_drop'])}
        for q in d.index}
SUMMARY['availability_shapley_top3_stack'] = SHAP_TABLE[['feature', 'shapley_stack', 'share_stack']].head(3).to_dict('records')
json.dump(SUMMARY, open(f'{OUT}/summary.json', 'w'), indent=2, default=float)

for name in DEPLOYMENTS:
    s = SUMMARY['deployments'][name]
    print(f"{name}: S0 {s['S0']['estimate']:.4f} -> S2 {s['S2']['estimate']:.4f} (drop {s['total_drop']['estimate']:.4f})")
    for q in ['availability_shapley', 'covariate_shapley', 'conditional']:
        e = s[q]
        print(f"   {q:22s} {e['estimate']:+.4f}  [{e['ci95'][0]:+.4f}, {e['ci95'][1]:+.4f}]  share {e['share_of_total_drop']:.2f}")
print('reproduction gate passed:', GATE['passed'])
print('weights balance ok:', WEIGHT_SUMMARY['balance_ok'], '| ESS positives', round(WEIGHT_SUMMARY['ess_positives_2015_test']))
print('outcome lineage:', LINEAGE.get('status'), '| same construct both years:', LINEAGE.get('same_construct_both_years'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.chdir('/content/drive/MyDrive/EXHEART_Research/exheart-research')

r = subprocess.run(["python", "tools/commit_cell.py",
  "17: same-model controls for the 2015-2020 sensitivity drop (availability, shared-predictor shift, outcome change, encoding counterfactuals, availability Shapley, raw CDC outcome lineage)"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)